# Tratamento da base de dados 

### Insira a base desejada para o tratamento:

In [ ]:
df_teste = 'nova_amostra.csv '  # insira o arquivo em formato csv no espaço entre as aspas simples (ex: 'teste_kaggle.csv)

1. Importando as bibliotecas necessárias para o tratamento

In [23]:
import pandas as pd   # biblioteca necessária para o tratamento dos dados em tabela
import numpy as np    # biblioteca necessária para o tratamento dados numéricos e para cálculos

2. Lendo as bases de dados necessárias para o tratamento

In [27]:
df_category = pd.read_csv('category.csv')
df_customers = pd.read_csv('customers.csv')
df_geolocation = pd.read_csv('geolocation.csv')
df_order_items = pd.read_csv('order_items.csv')
df_order_payments = pd.read_csv('order_payments.csv')
df_order_reviews = pd.read_csv('order_reviews.csv')
df_products = pd.read_csv('products.csv')
df_sellers = pd.read_csv('sellers.csv')

#colunas = ['order_id','customer_id','order_purchase_timestamp','order_approved_at']

df_teste = pd.read_csv('nova_amostra.csv').drop(columns=['customer_unique', 'customer_city', 'customer_state', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value','payment_sequential','payment_type','payment_installments','payment_value'])
df_teste['customer_id'] = df_teste['customer_id'].str.replace('\n', '', regex=True)
df_teste['customer_id'] = df_teste['customer_id'].str.replace('\r', '', regex=True)
df_teste.to_csv('test.csv')
df_teste.head()

,customer_id,customer_zip_code,order_id,order_purchase_timestamp,order_approved_at
0,06b8999e2fba1a1fbc88172c00ca8ac7,20231,12018f77f2f0320c557190d7a144bdd3,2024-07-07 10:56:33,2024-07-08 12:56:33
1,06b1299e2fba1a1fbc88172c00ca8ac7,5704,32018f77f2f0320c557190d7a144bdd3,2024-07-07 10:56:33,7/8/2024 12:56:33
2,16b1499e2fba1a1fbc88172c00ca8ac7,95110,54018f77f2f0320c557190d7a144bdd3,2024-07-07 10:56:33,7/9/2024 12:00:33
3,17b1499e2fba1a1fbc88172c00ca8ac7,13412,42018f77f2f0320c557190d7a144bdd3,2024-07-07 10:56:33,7/8/2024 21:56:33
4,18b1299e2fba1a1fbc88172c00ca8ac7,22750,54018f77f2f0320c557190d7a144bdd3,2024-07-07 10:56:33,7/7/2024 12:56:33


3. Agrupando todas as informações em uma mesma base de dados

In [ ]:
# Mesclar dados de teste com dados de clientes
df_geolocationt = df_geolocation.rename(columns={
    'geolocation_zip_code_prefix': 'customer_zip_code',
})

if 'customer_id' in df_teste.columns:
    df = df_teste.merge(df_customers)
else:
    df = df_teste.merge(df_geolocationt, on='customer_zip_code', how='left')

# Renomear colunas de geolocalização para clientes
df_geolocation_c = df_geolocation.rename(columns={
    'geolocation_zip_code_prefix': 'customer_zip_code_prefix',
    'geolocation_lat': 'geolocation_lat_c',
    'geolocation_lng': 'geolocation_lng_c',
    'geolocation_city': 'geolocation_city_c',
    'geolocation_state': 'geolocation_state_c'
})

# Renomear colunas de geolocalização para vendedores
df_geolocation_s = df_geolocation.rename(columns={
    'geolocation_zip_code_prefix': 'seller_zip_code_prefix',
    'geolocation_lat': 'geolocation_lat_s',
    'geolocation_lng': 'geolocation_lng_s',
    'geolocation_city': 'geolocation_city_s',
    'geolocation_state': 'geolocation_state_s'
})
df.info()
df.head()

In [26]:
# Mesclar dados de teste com dados de clientes
df = df_teste.merge(df_customers)

# Renomear colunas de geolocalização para clientes
df_geolocation_c = df_geolocation.rename(columns={
    'geolocation_zip_code_prefix': 'customer_zip_code_prefix',
    'geolocation_lat': 'geolocation_lat_c',
    'geolocation_lng': 'geolocation_lng_c',
    'geolocation_city': 'geolocation_city_c',
    'geolocation_state': 'geolocation_state_c'
})

# Renomear colunas de geolocalização para vendedores
df_geolocation_s = df_geolocation.rename(columns={
    'geolocation_zip_code_prefix': 'seller_zip_code_prefix',
    'geolocation_lat': 'geolocation_lat_s',
    'geolocation_lng': 'geolocation_lng_s',
    'geolocation_city': 'geolocation_city_s',
    'geolocation_state': 'geolocation_state_s'
})

# Remover duplicatas de geolocalização de clientes e mesclar com o dataframe principal
df_geolocation_c = df_geolocation_c.drop_duplicates()
df = df.merge(df_geolocation_c, on='customer_zip_code_prefix', how='left')

# Remover duplicatas de pagamentos por pedido e mesclar com o dataframe principal
df_order_payments = df_order_payments.drop_duplicates(subset=['order_id'])
df = df.merge(df_order_payments, on='order_id', how='left')

# Remover duplicatas de vendedores e mesclar com itens de pedido
df_sellers = df_sellers.drop_duplicates(subset=['seller_id'])
df2 = df_order_items.merge(df_sellers, on='seller_id', how='left')

# Remover duplicatas de produtos e mesclar com o dataframe de itens de pedido
df_products = df_products.drop_duplicates(subset=['product_id'])
df2 = df2.merge(df_products, on='product_id', how='left')

# Remover duplicatas de geolocalização de vendedores e mesclar com o dataframe de itens de pedido
df_geolocation_s = df_geolocation_s.drop_duplicates()
df2 = df2.merge(df_geolocation_s, on='seller_zip_code_prefix', how='left')

df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12250774 entries, 0 to 12250773
Data columns (total 22 columns):
 #   Column                      Dtype  
---  ------                      -----  
 0   order_id                    object 
 1   order_item_id               int64  
 2   product_id                  object 
 3   seller_id                   object 
 4   shipping_limit_date         object 
 5   price                       float64
 6   freight_value               float64
 7   seller_zip_code_prefix      int64  
 8   seller_city                 object 
 9   seller_state                object 
 10  product_category_name       object 
 11  product_name_lenght         float64
 12  product_description_lenght  float64
 13  product_photos_qty          float64
 14  product_weight_g            float64
 15  product_length_cm           float64
 16  product_height_cm           float64
 17  product_width_cm            float64
 18  geolocation_lat_s           float64
 19  geolocation_lng_s  

4. Tratando as variáveis, criando novas e descartandoa as inúteis

In [5]:
# Calcular o volume do produto
df = df2
df['product_volume'] = df['product_length_cm'] * df['product_height_cm'] * df['product_width_cm']

# Função para calcular a distância usando a fórmula de Haversine
def calcular_distancia(lat1, lon1, lat2, lon2):
    # Converter as coordenadas de graus para radianos
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    
    # Diferenças das coordenadas
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    
    # Fórmula de Haversine
    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    
    # Raio da Terra em quilômetros
    raio_terra_km = 6371
    distancia = raio_terra_km * c
    return distancia

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12250774 entries, 0 to 12250773
Data columns (total 23 columns):
 #   Column                      Dtype  
---  ------                      -----  
 0   order_id                    object 
 1   order_item_id               int64  
 2   product_id                  object 
 3   seller_id                   object 
 4   shipping_limit_date         object 
 5   price                       float64
 6   freight_value               float64
 7   seller_zip_code_prefix      int64  
 8   seller_city                 object 
 9   seller_state                object 
 10  product_category_name       object 
 11  product_name_lenght         float64
 12  product_description_lenght  float64
 13  product_photos_qty          float64
 14  product_weight_g            float64
 15  product_length_cm           float64
 16  product_height_cm           float64
 17  product_width_cm            float64
 18  geolocation_lat_s           float64
 19  geolocation_lng_s  

In [6]:

# Aplicar a função ao DataFrame para criar a nova coluna com as distâncias
df['distancia_km'] = df.apply(lambda row: calcular_distancia(row['geolocation_lat_c'], row['geolocation_lng_c'], row['geolocation_lat_s'], row['geolocation_lng_s']), axis=1)
df = df.drop(columns=['geolocation_lat_c', 'geolocation_lng_c', 'geolocation_lat_s', 'geolocation_lng_s'])

# Converter colunas de timestamp para datetime
df['order_purchase_timestamp'] = pd.to_datetime(df['order_purchase_timestamp'])
df['order_approved_at'] = pd.to_datetime(df['order_approved_at'])

# Calcular o tempo de aprovação da compra em minutos
df['purchase_to_approval_minutes'] = (df['order_approved_at'] - df['order_purchase_timestamp']).dt.total_seconds() / 86400

# Dicionário de substituições para padronizar variações de nomes comuns
substituicoes = {
    "sp": "sao paulo",
    "sao paulo": "sao paulo",
    "so paulo": "sao paulo",
    "Sao paulo": "sao paulo",
    "saopaulo": "sao paulo",
    "são paulo": "sao paulo",
    "saopaulo": "sao paulo",
    "varzea paulista": "varzea paulista"
    # Adicione outras variações de cidade aqui, se necessário
}

# Função para aplicar as substituições específicas
def padronizar_cidade(nome):
    return substituicoes.get(nome, nome)  # Substitui se a cidade estiver no dicionário, caso contrário, mantém o original

# Aplicando a normalização e padronização nas colunas de cidades
df['geolocation_city_c'] = df['geolocation_city_c'].apply(padronizar_cidade)
df['geolocation_city_s'] = df['geolocation_city_s'].apply(padronizar_cidade)

KeyError: 'geolocation_lat_c'

5. Organizando o datframe de forma que facilite a vizualização

In [ ]:
# Remover duplicatas com base em 'order_id' e 'product_id'
df_teste = df.drop_duplicates(subset=['order_id', 'product_id'])

# Ordenar o DataFrame por 'order_id', 'distancia_km', 'purchase_to_approval_minutes', 'freight_value' e 'product_volume' em ordem decrescente
df_teste = df_teste.sort_values(
    by=['order_id', 'distancia_km', 'purchase_to_approval_minutes', 'freight_value', 'product_volume'], 
    ascending=[False, False, False, False, False]
)

# Manter apenas a linha com os maiores valores por 'order_id'
df_teste = df_teste.groupby('order_id').first().reset_index()

6. Retirando as colunas já usadas para o tratamento que não serão necessárias para a previsão

In [ ]:
# Renomear a coluna 'purchase_to_approval_minutes' para 'purchase_to_approval_days'
df_teste.rename(columns={'purchase_to_approval_minutes': 'purchase_to_approval_days'}, inplace=True)

# Remover as colunas especificadas do DataFrame
df_teste = df_teste.drop(columns=[
    'customer_id',
    'order_purchase_timestamp',
    'order_approved_at',
    'customer_zip_code_prefix',
    'order_item_id',
    'product_id',
    'seller_id',
    'shipping_limit_date',
    'seller_zip_code_prefix'
])

7. Importação da base de teste já tratada em formato .csv

In [ ]:
df_teste.to_csv('Teste_Tratado.csv', index=False)